In [1]:
import sklearn
import numpy
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import kaggle
from tabulate import tabulate

In [2]:
from pathlib import Path

base = Path("./enron_mail_20150507/maildir/allen-p/_sent_mail")
print(list(base.iterdir())[:10])

[WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/10.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/100.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1000.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1001.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1002.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1003.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/1004.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/101.'), WindowsPath('enron_mail_20150507/maildir/allen-p/_sent_mail/102.')]


In [3]:
from pathlib import Path
import shutil
import os

#RAW_ROOT = Path(r"********CURRENT DATA FOLDER*******")
#WIN_ROOT = Path(r"******YOUR NEW FOLDER HERE*******")

def make_windows_safe_name(name: str) -> str:
    """Convert '123.' -> '123.txt'"""
    return name.rstrip(".") + ".txt"

def copy_dataset_safe(raw_root: Path, win_root: Path):
    win_root.mkdir(parents=True, exist_ok=True)

    for src_path in raw_root.iterdir():

        safe_name = make_windows_safe_name(src_path.name)
        dst_file = win_root / safe_name
        
        if dst_file.exists():
            continue
        
        if src_path.is_dir():
            copy_dataset_safe(src_path, win_root / src_path.name)    
        else:
           # Windows-safe name
            dst_file.parent.mkdir(parents=True, exist_ok=True)

            # Use Windows UNC prefix to reliably open files ending with dot
            src_path_unc = Path(r"\\?\\" + str(src_path.resolve()))

            try:
                with open(src_path_unc, "r", encoding="latin-1", errors="ignore") as f_src:
                    content = f_src.read()
                with open(dst_file, "w", encoding="latin-1", errors="ignore") as f_dst:
                    f_dst.write(content)
            except Exception as e:
                print(f"⚠️ Skipped {src_path}: {e}") 
            
# Run copy function to convert dataset from UNIX -> Windows .txt files
#copy_dataset_safe(RAW_ROOT, WIN_ROOT)
print("Windows-safe Enron dataset created successfully.")


Windows-safe Enron dataset created successfully.


In [4]:
import re

def remove_reply_chain(text):

    reply_patterns = [
        r"-----Original Message-----",
        r"----- Forwarded by",
        r"\s+From: .*",
        r"\s+Sent: .*",
        r"\s+To: .*",
        r"\s+Subject: .*",
        r"On .* wrote:",
        r"\S+@\S+\s+on\s+\d{1,2}/\d{1,2}/\d{4}" # Looking for email@address on D/M/Y 
    ]

    for pattern in reply_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            text = text[:match.start()]

    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [5]:
from email import policy
from email.parser import Parser

def parse_email(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        msg = Parser(policy=policy.default).parse(f)

    data = {
        "message_id": msg["Message-ID"],
        "date": msg["Date"],
        "from": msg["From"],
        "to": msg["To"],
        "cc": msg["Cc"],
        "bcc": msg["Bcc"],
        "subject": msg["Subject"],
        "body": msg.get_body(preferencelist=('plain')).get_content()
            if msg.get_body()
            else msg.get_payload()
    }

    data["body"] = remove_reply_chain(data["body"])
    
    return data
    
print(parse_email("./enron_windows_dataset/arnold-j/_sent_mail/111.txt"))

{'message_id': '<22562856.1075857596586.JavaMail.evans@thyme>', 'date': 'Sat, 14 Oct 2000 10:00:00 -0700', 'from': 'john.arnold@enron.com', 'to': 'slafontaine@globalp.com', 'cc': None, 'bcc': None, 'subject': 'Re: mkts', 'body': "pira's certainly got the whole market wound up. I've seen a wave of producer selling for the first time in two months over the past two days. Most selling Cal 1 off the back of Pira. Pira certainly commands a lot of respect these days. Too much, probably. The problem with all these bull spreads (ie F-H) is the thought process in natty is that if Jan is strong, just think what happens when you get to March and run out of gas. The spread game is very different than playing crude. These spreads haven't moved for the past 1000 point runup. You know there were guys bullish this market trying to play it with spreads and haven't made a penny. Just to clarify, Pira said 3 bcf y on y for Z1? That seems hard to believe."}
